# Strict run-wise split with non-overlapping windows


**Important:** Place `merged_physics.csv` and `uav_strict_utils.py` in the same folder as this notebook before running.  
The notebook saves `metrics.csv`, `classification_report.txt`, `predictions.csv`, `confusion_matrix.pdf`, `metrics_bar.pdf`, `training_curve.pdf`, and the trained `.keras` model for each technique.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from uav_strict_utils import *

INPUT_FILE = 'merged_physics.csv'
RESULT_ROOT = Path('Strict_Runwise_Results')
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 50
STRIDE = 50
EPOCHS = 80
BATCH_SIZE = 16

# Prepare non-overlapping windows after removing idle RPM region
_, X, y, y_text, groups, label_encoder, feature_cols = prepare_windows(
    INPUT_FILE, window_size=WINDOW_SIZE, stride=STRIDE, filter_active=True
)

# Split complete source files, not random windows
file_labels = pd.DataFrame({'source_file': groups, 'label': y_text}).drop_duplicates()
file_majority = file_labels.groupby('source_file')['label'].agg(lambda s: s.mode()[0]).reset_index()
files = file_majority['source_file'].values
file_y = file_majority['label'].values

trainval_files, test_files, trainval_labels, test_labels = train_test_split(
    files, file_y, test_size=0.33, stratify=file_y, random_state=SEED
)
try:
    train_files, val_files, train_labels, val_labels = train_test_split(
        trainval_files, trainval_labels, test_size=0.33, stratify=trainval_labels, random_state=SEED
    )
except ValueError:
    train_files, val_files, train_labels, val_labels = train_test_split(
        trainval_files, trainval_labels, test_size=0.33, random_state=SEED
    )

train_idx = np.where(np.isin(groups, train_files))[0]
val_idx = np.where(np.isin(groups, val_files))[0]
test_idx = np.where(np.isin(groups, test_files))[0]

print('Train files:', train_files)
print('Validation files:', val_files)
print('Test files:', test_files)

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
X_train, X_val, X_test, scaler = scale_by_train(X_train, X_val, X_test)

print('Train:', X_train.shape, pd.Series(label_encoder.inverse_transform(y_train)).value_counts().to_dict())
print('Validation:', X_val.shape, pd.Series(label_encoder.inverse_transform(y_val)).value_counts().to_dict())
print('Test:', X_test.shape, pd.Series(label_encoder.inverse_transform(y_test)).value_counts().to_dict())

all_metrics = []
for model_name in MODEL_NAMES:
    metrics = train_one_model(
        model_name, X_train, y_train, X_val, y_val, X_test, y_test,
        label_encoder, feature_cols, RESULT_ROOT / model_name,
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle_labels=False
    )
    metrics['Model'] = model_name
    all_metrics.append(metrics)

summary_df = pd.DataFrame(all_metrics)
summary_df = summary_df[['Model'] + [c for c in summary_df.columns if c != 'Model']]
summary_df.to_csv(RESULT_ROOT / 'all_models_summary.csv', index=False)
summary_df


C:\Users\Hp\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Idle rows removed: 47
Dataset shape after cleaning/filtering: (1396, 33)
Number of numerical features: 31
Class counts after filtering:
class_label
bend      591
normal    589
crack     216
Name: count, dtype: int64

Created non-overlapping windows
Total windows: 21
Window class counts:
bend      9
normal    9
crack     3
Name: count, dtype: int64
Source files: 9
Train files: ['Propeller_Data_1045_normal1.csv' 'Propeller_Data_1045_bend2.csv'
 'Propeller_Data_1045_normal2.csv' 'Propeller_Data_1045_crack2.csv']
Validation files: ['Propeller_Data_1045_crack3.csv' 'Propeller_Data_1045_bend1.csv']
Test files: ['Propeller_Data_1045_crack1.csv' 'Propeller_Data_1045_bend3.csv'
 'Propeller_Data_1045_normal3.csv']
Train: (10, 50, 31) {'normal': 6, 'bend': 3, 'crack': 1}
Validation: (4, 50, 31) {'bend': 3, 'crack': 1}
Test: (7, 50, 31) {'bend': 3, 'normal': 3, 'crack': 1}

Epoch 1/80
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.3000 - loss: 1.1727 - val_accuracy: 0.0000e+00 - val_loss: 1.083

,Model,Accuracy,Precision,Recall,F1Score,Precision_weighted,Recall_weighted,F1Score_weighted
0,1D_CNN,0.285714,0.194444,0.222222,0.206349,0.250000,0.285714,0.265306
1,LSTM,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,CNN_LSTM,0.714286,0.472222,0.555556,0.507937,0.607143,0.714286,0.653061
3,TCN,0.428571,0.142857,0.333333,0.200000,0.183673,0.428571,0.257143
4,Transformer,0.857143,0.583333,0.666667,0.619048,0.750000,0.857143,0.795918
5,Multimodal_Transformer,0.857143,0.583333,0.666667,0.619048,0.750000,0.857143,0.795918
